### Create schema and volume for project

In [0]:
%sql
create schema if not exists main.lufthansa_level3;
create volume if not exists main.lufthansa_level3.staging_area;

CREATE SCHEMA IF NOT EXISTS main.lufthansa_bronze;
CREATE SCHEMA IF NOT EXISTS main.lufthansa_silver;
CREATE SCHEMA IF NOT EXISTS main.lufthansa_gold;
CREATE VOLUME IF NOT EXISTS main.lufthansa_bronze.landing_area;

In [0]:
%run ../utils/nets

In [0]:
%run ../utils/save_utils

In [0]:
def get_and_save_data(endpoint):
    response = get_data(endpoint)
    data = response.json()
    full_save_path = get_full_save_path(endpoint, timestamp=True)
    save_json_with_dbutils(data, full_save_path, True, 2)

In [0]:
def get_and_save_one(endpoint, path_params=None, query_params=None):
    response = fetch_data(
        endpoint,
        path_params=path_params,
        query_params=query_params,
    )

    data = response.json()

    full_save_path = get_full_save_path(
        endpoint=endpoint,
        timestamp=True,
    )

    save_json_with_dbutils(data, full_save_path, True, 2)

In [0]:
def get_total_count(data):
    resource = next(iter(data.values()), {})
    return resource.get("Meta", {}).get("TotalCount")

In [0]:
def get_and_save_all_pages(
    endpoint,
    path_params=None,
    query_params=None,
    limit=100,
    time_dir_format=None
):
    if query_params:
        query_params = query_params.copy() 
    else:
        query_params = {}

    offset = 10900
    total = None

    while True:
        page_query_params = {
            **query_params,
            "limit": limit,
            "offset": offset,
        }

        response = fetch_data(
            endpoint,
            path_params=path_params,
            query_params=page_query_params,
        )

        data = response.json()
        print(f"Loading page {offset} from {total}")
        if total is None:
            total = get_total_count(data)
            if total is None:
                raise ValueError("Response does not contain 'total'")

        # items = data.get("data")
        # if items is None:
        #     raise ValueError("Response does not contain 'data'")
        time_dir = datetime.datetime.now().strftime(time_dir_format)
        print(time_dir)
        full_save_path = get_full_save_path(
            endpoint=endpoint,
            offset=offset,
            time_dir=time_dir
        )
        print(full_save_path)

        save_json_with_dbutils(data, full_save_path, True, 2)

        offset += limit

        if offset >= total:
            break

        # if not items:
        #     raise ValueError(
        #         f"Empty page received before reaching total. offset={offset}, total={total}"
        #     )

In [0]:
def get_and_save_data(endpoint, path_params=None, query_params=None, limit=100):
    if endpoint in LIST_ENDPOINTS:
        return get_and_save_all_pages(
            endpoint=endpoint,
            path_params=path_params,
            query_params=query_params,
            limit=limit,
        )

    return get_and_save_one(
        endpoint=endpoint,
        path_params=path_params,
        query_params=query_params,
    )

In [0]:
%run ../utils/ingestion_utils

In [0]:


# get_and_save_data(EndpointKeys.COUNTRIES)
# get_and_save_data(EndpointKeys.CITIES)
# get_and_save_data(EndpointKeys.AIRPORTS)
# get_and_save_data(EndpointKeys.AIRLINES)
# get_and_save_data(EndpointKeys.AIRCRAFT)
# response = get_flights("FRA", "HAM", "2026-03-10")

# response = fetch_data(EndpointKeys.AIRPORTS)
# path_params = {
#     "origin": "FRA",
#     "destination": "HAM",
#     "date": "2026-03-10",
# }
# response = fetch_data(EndpointKeys.FLIGHTSTATUS_BY_ROUTE, path_params=path_params)

# full_save_path = get_full_save_path(endpoint=EndpointKeys.COUNTRIES, timestamp=True,
#                                     offset=100, ts_format=PROFILE.get(ProfileKeys.TS_MONTH_FORMAT))
# print(full_save_path)

response = fetch_data(EndpointKeys.COUNTRIES)
data = response.json()
print(get_total_count(data))
print(data.get("CountryResource").get("Meta").get("TotalCount"))

# get_and_save_all_pages(EndpointKeys.AIRPORTS, limit=100, time_dir_format=PROFILE.get(ProfileKeys.TS_MONTH_FORMAT))

get_and_save_all_pages(EndpointKeys.AIRPORTS, limit=100, time_dir_format=PROFILE.get(ProfileKeys.TS_DAYLY_FORMAT))

# get_and_save_one(
#     EndpointKeys.AIRPORT_CODE,
#     path_params={"airportCode": "FRA"},
# )
# source_path = "/Volumes/main/lufthansa_level3/staging_area/tmp/references/airports"
# target_table ="main.lufthansa_level3.bronze_airports"
# load_json_to_bronze(source_path, target_table)

In [0]:
# get_and_save_all_pages(
#     EndpointKeys.AIRPORTS,
#     query_params={},
#     limit=100,
# )

In [0]:
# load_data_to_bronze(EndpointKeys.CITIES)
# load_countries_to_bronze(EndpointKeys.COUNTRIES)

In [0]:
%sql
-- select * from main.lufthansa_bronze.countries
drop table main.lufthansa_bronze.countries;
drop table main.lufthansa_bronze.airports;

In [0]:
%sql
-- select * from main.lufthansa_level3.bronze_airports where `_source_file` like '%20260310%'


In [0]:
restart = False
if restart:
    dbutils.library.restartPython()

In [0]:
%sql
select * from main.lufthansa_bronze.countries